# Paper 4 — End-to-End Free-GPU Development Notebook

**Research direction:** Evidence verification, calibration, and selective answering for KB-VQA.

This notebook is development-first. It must not be used to manufacture paper numbers. `DEV_MODE=True` runs a small A-OKVQA validation subset. Final experiments require a locked protocol and fresh held-out evaluation.


In [ ]:
import os, sys, json, platform, subprocess, pathlib, time
DEV_MODE = True
MAX_SAMPLES = 100 if DEV_MODE else None
REPO_URL = "https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git"
ROOT = pathlib.Path("/content/paper4-selective-kbvqa") if pathlib.Path("/content").exists() else pathlib.Path("./paper4-selective-kbvqa")
print({"DEV_MODE": DEV_MODE, "MAX_SAMPLES": MAX_SAMPLES, "ROOT": str(ROOT)})


## 1. Acquire repository

Once the dedicated GitHub repository exists, this cell clones it. If the notebook was uploaded inside the project directory, cloning is skipped.


In [ ]:
if not (ROOT / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print("cwd=", os.getcwd())


## 2. Install free/open dependencies


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[vlm,retrieval,dev]"], check=True)


## 3. Hardware audit — hard gate

The benchmark path must run on a real CUDA GPU. This cell records the exact environment.


In [ ]:
import torch
from paper4_kbvqa.utils.environment import write_environment
print(write_environment("results/environment.json"))
assert torch.cuda.is_available(), "CUDA GPU is required for VLM benchmark execution"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,2), "GB")


## 4. Run unit tests before any expensive inference


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## 5. Download A-OKVQA annotations + COCO 2017 validation images for DEV_MODE

The A-OKVQA commands follow the official repository. DEV_MODE downloads only validation images to keep free-session storage manageable.


In [ ]:
AOK = ROOT / "datasets/aokvqa"
COCO = ROOT / "datasets/coco"
AOK.mkdir(parents=True, exist_ok=True); COCO.mkdir(parents=True, exist_ok=True)
archive=AOK/"aokvqa_v1p0.tar.gz"
if not (AOK/"aokvqa_v1p0_val.json").exists():
    subprocess.run(["curl","-fL","https://prior-datasets.s3.us-east-2.amazonaws.com/aokvqa/aokvqa_v1p0.tar.gz","-o",str(archive)],check=True)
    subprocess.run(["tar","-xzf",str(archive),"-C",str(AOK)],check=True)
valzip=COCO/"val2017.zip"
if not (COCO/"val2017").exists():
    subprocess.run(["wget","-q","http://images.cocodataset.org/zips/val2017.zip","-O",str(valzip)],check=True)
    subprocess.run(["unzip","-q",str(valzip),"-d",str(COCO)],check=True)
print("dataset ready")


## 6. Build leakage-safe manifest

Ground-truth answers stay in evaluation fields. `VQASample.inference_view()` prevents them from entering retrieval or generation.


In [ ]:
manifest=ROOT/"datasets/manifests/aokvqa_val.jsonl"
manifest.parent.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,"scripts/build_manifest.py","aokvqa","--aokvqa-dir",str(AOK),"--coco-dir",str(COCO),"--split","val","--out",str(manifest)],check=True)
print(manifest)


## 7. B0 — VLM without external knowledge


In [ ]:
cmd=[sys.executable,"scripts/run_baseline.py","--manifest",str(manifest),"--baseline","B0","--out","results/predictions/aokvqa_val_B0.jsonl","--checkpoint","results/checkpoints/aokvqa_val_B0.json","--run-id","aokvqa-val-B0"]
if MAX_SAMPLES: cmd += ["--max-samples",str(MAX_SAMPLES)]
subprocess.run(cmd,check=True)


## 8. B1 — raw multi-source knowledge


In [ ]:
cmd=[sys.executable,"scripts/run_baseline.py","--manifest",str(manifest),"--baseline","B1","--out","results/predictions/aokvqa_val_B1.jsonl","--checkpoint","results/checkpoints/aokvqa_val_B1.json","--run-id","aokvqa-val-B1","--sources","wikipedia,wikidata,conceptnet"]
if MAX_SAMPLES: cmd += ["--max-samples",str(MAX_SAMPLES)]
subprocess.run(cmd,check=True)


## 9. B2/B3 — filtered knowledge + evidence verification


In [ ]:
cmd=[sys.executable,"scripts/run_proposed.py","--manifest",str(manifest),"--out","results/predictions/aokvqa_val_proposed.jsonl","--checkpoint","results/checkpoints/aokvqa_val_proposed.json","--run-id","aokvqa-val-proposed","--sources","wikipedia,wikidata,conceptnet","--top-k","5"]
if MAX_SAMPLES: cmd += ["--max-samples",str(MAX_SAMPLES)]
subprocess.run(cmd,check=True)


## 10. Calibration/selective policy — DEVELOPMENT ONLY here

For a publication run, fit the calibration/threshold policy on a **locked validation/calibration split** and apply it unchanged to a separate held-out evaluation split. Do not fit and report on the same records.


In [ ]:
if DEV_MODE:
    print("Calibration policy fitting intentionally skipped in DEV_MODE to avoid reporting same-split selective metrics as final.")
else:
    print("Use scripts/fit_selective_policy.py on the locked calibration split, then scripts/apply_selective_policy.py on held-out predictions.")


## 11. Integrity checkpoint


In [ ]:
print("DEV outputs are engineering evidence only." if DEV_MODE else "Before manuscript use, verify split separation, official evaluator, baselines, ablations, statistics, and reproducibility audit.")
